In [84]:
#Importacion de librerias
import requests
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
from googleapiclient.discovery import build
from google.oauth2 import service_account


#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [3]:
df_excel = pd.read_excel(r'C:\Users\alejandro_arboledas\Downloads\MERCADEO_Y_PUBLICIDAD.xlsx', sheet_name='MERCADEO')
df_excel.shape

(29474, 3)

In [4]:
para_orden = df_excel['DOC_ALUM'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
periodos_unicos = df_excel['PERIODO'].unique()
valores_unicos_periodos = "', '".join(map(str, periodos_unicos))
valores_unicos_periodos = f"'{valores_unicos_periodos}'"

In [47]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' select * from cun_repositorio.cun.DescuentosBeca
        where identificacion in ({valores_unicos_df1_orden})
        and NOMBRE_GRUPO = 'BECAS'
                        ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_pre = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_pre.shape

(225, 6)

In [48]:
df_pre

,PERIODO,NOMBRE_GRUPO,IDENTIFICACION,NOMBRE_CLIENTE,NOMBRE_CONCEPTO,NOMBRE_CAUSA_NOTA
0,24V03,BECAS,1013615457,CARLOS ARMANDO GUEVARA FRANCO,NC BECAS,APOYO SOCIAL BECA
1,2017B,BECAS,1024507984,HAROLD EDUARDO AMAYA GONZALEZ,NC BECAS,DEPORTIVA BECA
2,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA
3,2019A,BECAS,1001294439,PATRICK YASIR ULLOA CASTRO,NC BECAS,APOYO SOCIAL BECA
4,2025B,BECAS,1012407209,SEBASTIAN FELIPE MARTINEZ RESTREPO,NC BECAS,APOYO SOCIAL BECA
...,...,...,...,...,...,...
220,2026A,BECAS,1054708141,BRAYAN FERNEY CASTRO ESPINOSA,NC BECAS,APOYO SOCIAL BECA
221,2016B,BECAS,80312242,PEDRO PABLO RIAÑO MORALES,NC BECAS,DEPORTIVA BECA
222,2019A,BECAS,1019139955,LAURA CAMILA TENJO NAVARRETE,NC BECAS,DEPORTIVA BECA
223,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA


In [52]:
df_pre_2 = df_pre.copy()
df_pre_2.drop_duplicates(inplace=True)
df_pre_2.shape

(225, 6)

In [ ]:
df_pre_2['NOMBRE_CAUSA_NOTA'].unique()

array(['APOYO SOCIAL BECA', 'DEPORTIVA BECA', 'EXCELENCIA ACADEMICA BECA',
       'DESCUENTOS CONVOCATORIA BLACK FRIDAY', 'BECA 100%',
       'CULTURAL BECA'], dtype=object)

In [30]:
para_orden = df_pre_2['IDENTIFICACION'].unique()
valores_unicos= "','".join(map(str, para_orden))
valores_unicos = f"'{valores_unicos}'"
periodos_unicos = df_pre_2['PERIODO'].unique()
valores_PERIODOS = "','".join(map(str, periodos_unicos))
valores_PERIODOS = f"'{valores_PERIODOS}'"

In [66]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' SELECT CLIENTE, PERIODO AS PERIODO_X, VALOR_BRUTO, DESCRIPCION FROM openquery([172.16.1.175],'
            select * from ICEBERG.Nota_credito
	        where CONCEPTO_NOTA = ''201''
            and causa_nota in (''16'',''201'',''202'',''203'',''204'',''216'') ')
            where cliente in ({valores_unicos})            
                        ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_nota = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_nota.shape

(172, 4)

In [69]:
df_notas_2 = df_nota.copy()

In [70]:
df_notas_2

,CLIENTE,PERIODO_X,VALOR_BRUTO,DESCRIPCION
0,1.001044e+09,2020A,784449.0,DESCUENTO BLACKWEEK-CAMBIO DE TARIFA
1,1.022404e+09,2020A,784449.0,APROBADO BLACK WEEK CAMBIO DE TARIFA // JAIME ...
2,1.016095e+09,2020A,784449.0,DESCUENTO BLACKWEEK-CAMBIO DE TARIFA
3,1.000494e+09,2020A,709450.0,APLICA DESCUENTO BLACK WEEK 35% ALEJANDRO LOPEZ
4,1.000020e+09,2020A,784449.0,APROBADO BLACK WEEK CAMBIO DE TARIFA // JAIME ...
...,...,...,...,...
167,1.070594e+09,2026A,2253280.0,DESCUENTO PRESENCIAL COMFACUNDI 2026A - SRODRI...
168,1.012917e+09,2026A,1408300.0,DESCUENTO CONVENIO NASES 50% #3168975 AUTORIZA...
169,1.031640e+09,2026A,704150.0,DESCUENTO JAC #3178153 - SRODRIGUEZ
170,1.054708e+09,2026A,1229190.0,SOLICTUD DE APLICACION DE DESCEUNTO POR CONVEN...


In [71]:

df_excel['LLAVE'] = df_excel['DOC_ALUM'].astype(str) 
df_pre_2['LLAVE'] = df_pre_2['IDENTIFICACION'].astype(str) 
df_notas_2['LLAVE'] = df_notas_2['CLIENTE'].astype("Int64").astype(str)

In [72]:
df_final = df_pre_2.merge(df_excel[['PROGRAMA','LLAVE']],how='inner', on='LLAVE')
df_final

,PERIODO,NOMBRE_GRUPO,IDENTIFICACION,NOMBRE_CLIENTE,NOMBRE_CONCEPTO,NOMBRE_CAUSA_NOTA,LLAVE,PROGRAMA
0,24V03,BECAS,1013615457,CARLOS ARMANDO GUEVARA FRANCO,NC BECAS,APOYO SOCIAL BECA,1013615457,PUBLICIDAD Y MERCADEO
1,2017B,BECAS,1024507984,HAROLD EDUARDO AMAYA GONZALEZ,NC BECAS,DEPORTIVA BECA,1024507984,PUBLICIDAD Y MERCADEO
2,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA
3,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA
4,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA
...,...,...,...,...,...,...,...,...
928,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA,1000327422,INGENIERIA ELECTRONICA
929,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA,1000327422,INGENIERIA ELECTRONICA
930,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA,1000327422,INGENIERIA ELECTRONICA
931,24V03,BECAS,1036404411,ALEXANDRA VARGAS CASTRILLON,NC BECAS,APOYO SOCIAL BECA,1036404411,PUBLICIDAD Y MERCADEO


In [73]:
df_final_2 =  df_final.merge(df_notas_2[['VALOR_BRUTO','DESCRIPCION','LLAVE', 'PERIODO_X']],how='inner', on='LLAVE')
df_final_2

,PERIODO,NOMBRE_GRUPO,IDENTIFICACION,NOMBRE_CLIENTE,NOMBRE_CONCEPTO,NOMBRE_CAUSA_NOTA,LLAVE,PROGRAMA,VALOR_BRUTO,DESCRIPCION,PERIODO_X
0,24V03,BECAS,1013615457,CARLOS ARMANDO GUEVARA FRANCO,NC BECAS,APOYO SOCIAL BECA,1013615457,PUBLICIDAD Y MERCADEO,461700.0,SE APLICA DESCUENTO POR CONVENIO VIGENTE BAJO ...,24V03
1,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA,1425160.0,SE GENERA NOTA POR CONVENIO FONDECOMPUTO // TI...,2025C
2,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA,1068800.0,SE GENERA DESCUETNO POR FONDECOMPUTO #2089233/...,2024A
3,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA,1068800.0,SE GEENRA DESCUENTO POR FONDECOMPUTO TICKET #...,2025A
4,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA,1068800.0,SE GENERA DESCUETNO POR FONDECOMPUTO #2323966 ...,2024B
...,...,...,...,...,...,...,...,...,...,...,...
2173,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA,1000327422,INGENIERIA ELECTRONICA,604750.0,BECA POR APOYO SOCIAL TICKET #2039234 // ATRIANA,2024A
2174,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA,1000327422,INGENIERIA ELECTRONICA,890725.0,CONVENIO / JFORERO,2026A
2175,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA,1000327422,INGENIERIA ELECTRONICA,890725.0,APOYO SOCIAL BECA TICKET #2957727//MCANGULO,2025C
2176,24V03,BECAS,1036404411,ALEXANDRA VARGAS CASTRILLON,NC BECAS,APOYO SOCIAL BECA,1036404411,PUBLICIDAD Y MERCADEO,344700.0,SE APLICA DESCUENTO POR CONVENIO VIGENTE BAJO ...,24V03


In [74]:
df_final_2.drop_duplicates(inplace=True)
df_final_2

,PERIODO,NOMBRE_GRUPO,IDENTIFICACION,NOMBRE_CLIENTE,NOMBRE_CONCEPTO,NOMBRE_CAUSA_NOTA,LLAVE,PROGRAMA,VALOR_BRUTO,DESCRIPCION,PERIODO_X
0,24V03,BECAS,1013615457,CARLOS ARMANDO GUEVARA FRANCO,NC BECAS,APOYO SOCIAL BECA,1013615457,PUBLICIDAD Y MERCADEO,461700.0,SE APLICA DESCUENTO POR CONVENIO VIGENTE BAJO ...,24V03
1,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA,1425160.0,SE GENERA NOTA POR CONVENIO FONDECOMPUTO // TI...,2025C
2,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA,1068800.0,SE GENERA DESCUETNO POR FONDECOMPUTO #2089233/...,2024A
3,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA,1068800.0,SE GEENRA DESCUENTO POR FONDECOMPUTO TICKET #...,2025A
4,2023B,BECAS,1014291321,RICHAR JUNIOR ACOSTA GIL,NC BECAS,APOYO SOCIAL BECA,1014291321,INGENIERIA ELECTRONICA,1068800.0,SE GENERA DESCUETNO POR FONDECOMPUTO #2323966 ...,2024B
...,...,...,...,...,...,...,...,...,...,...,...
2147,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA,1000327422,INGENIERIA ELECTRONICA,668000.0,BECA POR APOYO SOCIAL TICKET #2324451//MCANGULO,2024B
2148,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA,1000327422,INGENIERIA ELECTRONICA,604750.0,BECA POR APOYO SOCIAL TICKET #2039234 // ATRIANA,2024A
2149,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA,1000327422,INGENIERIA ELECTRONICA,890725.0,CONVENIO / JFORERO,2026A
2150,2024B,BECAS,1000327422,KEVIN YESITH RUIZ AGUILAR,NC BECAS,APOYO SOCIAL BECA,1000327422,INGENIERIA ELECTRONICA,890725.0,APOYO SOCIAL BECA TICKET #2957727//MCANGULO,2025C


In [76]:
df_final_2.columns

Index(['PERIODO', 'NOMBRE_GRUPO', 'IDENTIFICACION', 'NOMBRE_CLIENTE',
       'NOMBRE_CONCEPTO', 'NOMBRE_CAUSA_NOTA', 'LLAVE', 'PROGRAMA',
       'VALOR_BRUTO', 'DESCRIPCION', 'PERIODO_X'],
      dtype='object')

In [77]:
df_final_3 = df_final_2[[ 'IDENTIFICACION','PERIODO_X','NOMBRE_CLIENTE',
        'NOMBRE_GRUPO','NOMBRE_CONCEPTO', 'NOMBRE_CAUSA_NOTA','PROGRAMA',
       'VALOR_BRUTO', 'DESCRIPCION']]
df_final_3

,IDENTIFICACION,PERIODO_X,NOMBRE_CLIENTE,NOMBRE_GRUPO,NOMBRE_CONCEPTO,NOMBRE_CAUSA_NOTA,PROGRAMA,VALOR_BRUTO,DESCRIPCION
0,1013615457,24V03,CARLOS ARMANDO GUEVARA FRANCO,BECAS,NC BECAS,APOYO SOCIAL BECA,PUBLICIDAD Y MERCADEO,461700.0,SE APLICA DESCUENTO POR CONVENIO VIGENTE BAJO ...
1,1014291321,2025C,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1425160.0,SE GENERA NOTA POR CONVENIO FONDECOMPUTO // TI...
2,1014291321,2024A,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1068800.0,SE GENERA DESCUETNO POR FONDECOMPUTO #2089233/...
3,1014291321,2025A,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1068800.0,SE GEENRA DESCUENTO POR FONDECOMPUTO TICKET #...
4,1014291321,2024B,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1068800.0,SE GENERA DESCUETNO POR FONDECOMPUTO #2323966 ...
...,...,...,...,...,...,...,...,...,...
2147,1000327422,2024B,KEVIN YESITH RUIZ AGUILAR,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,668000.0,BECA POR APOYO SOCIAL TICKET #2324451//MCANGULO
2148,1000327422,2024A,KEVIN YESITH RUIZ AGUILAR,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,604750.0,BECA POR APOYO SOCIAL TICKET #2039234 // ATRIANA
2149,1000327422,2026A,KEVIN YESITH RUIZ AGUILAR,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,890725.0,CONVENIO / JFORERO
2150,1000327422,2025C,KEVIN YESITH RUIZ AGUILAR,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,890725.0,APOYO SOCIAL BECA TICKET #2957727//MCANGULO


In [78]:
df_final_3.to_excel(r'C:\Users\alejandro_arboledas\Downloads\MERCADEO_Y_PUBLICIDAD_FINAL.xlsx')

In [80]:
df_base = pd.read_excel(r'C:\Users\alejandro_arboledas\Downloads\MERCADEO_Y_PUBLICIDAD_FINAL.xlsx')
df_base

,IDENTIFICACION,PERIODO,NOMBRE_CLIENTE,NOMBRE_GRUPO,NOMBRE_CONCEPTO,NOMBRE_CAUSA_NOTA,PROGRAMA,VALOR_BECA
0,1013615457,24V03,CARLOS ARMANDO GUEVARA FRANCO,BECAS,NC BECAS,APOYO SOCIAL BECA,PUBLICIDAD Y MERCADEO,461700
1,1014291321,2025C,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1425160
2,1014291321,2024A,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1068800
3,1014291321,2025A,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1068800
4,1014291321,2024B,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1068800
...,...,...,...,...,...,...,...,...
156,37083522,24V03,ANDREA JIMENA BURBANO Pasto,BECAS,NC BECAS,APOYO SOCIAL BECA,PUBLICIDAD Y MERCADEO,344700
157,1014481699,2025A,ANDREA KATHERIN GUTIERREZ AMAYA,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,801600
158,1012917427,2026A,NICOLLE GARZÓN COTERA,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1408300
159,1054708141,2026A,BRAYAN FERNEY CASTRO ESPINOSA,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1229190


In [83]:
para_orden = df_base['IDENTIFICACION'].unique()
valores_unicos_df1_orden = "', '".join(map(str, para_orden))
valores_unicos_df1_orden = f"'{valores_unicos_df1_orden}'"
periodos_unicos = df_base['PERIODO'].unique()
valores_unicos_periodos = "', '".join(map(str, periodos_unicos))
valores_unicos_periodos = f"'{valores_unicos_periodos}'"

In [85]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' SELECT ciclo_sugerido, Identificacion, Periodo from CUN_REPOSITORIO.Desercion.Python_CUN_Simulado_cambios_ciclo 
                            where identificacion in ({valores_unicos_df1_orden})        
                        ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_cambios = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_cambios.shape

(519, 3)

In [87]:
df_base['LLAVE'] = df_base['IDENTIFICACION'].astype(str) + df_base['PERIODO']
df_cambios['LLAVE'] = df_cambios['Identificacion'].astype(str) + df_cambios['Periodo']

In [91]:
df_final_final = df_base.merge(df_cambios[['ciclo_sugerido','LLAVE']],how='left',on='LLAVE')
df_final_final

,IDENTIFICACION,PERIODO,NOMBRE_CLIENTE,NOMBRE_GRUPO,NOMBRE_CONCEPTO,NOMBRE_CAUSA_NOTA,PROGRAMA,VALOR_BECA,LLAVE,ciclo_sugerido
0,1013615457,24V03,CARLOS ARMANDO GUEVARA FRANCO,BECAS,NC BECAS,APOYO SOCIAL BECA,PUBLICIDAD Y MERCADEO,461700,101361545724V03,TECNOLOGO
1,1014291321,2025C,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1425160,10142913212025C,TECNOLOGO
2,1014291321,2024A,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1068800,10142913212024A,TECNICO
3,1014291321,2025A,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1068800,10142913212025A,TECNICO
4,1014291321,2024B,RICHAR JUNIOR ACOSTA GIL,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1068800,10142913212024B,TECNICO
...,...,...,...,...,...,...,...,...,...,...
156,37083522,24V03,ANDREA JIMENA BURBANO Pasto,BECAS,NC BECAS,APOYO SOCIAL BECA,PUBLICIDAD Y MERCADEO,344700,3708352224V03,TECNICO
157,1014481699,2025A,ANDREA KATHERIN GUTIERREZ AMAYA,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,801600,10144816992025A,NaN
158,1012917427,2026A,NICOLLE GARZÓN COTERA,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1408300,10129174272026A,TECNICO
159,1054708141,2026A,BRAYAN FERNEY CASTRO ESPINOSA,BECAS,NC BECAS,APOYO SOCIAL BECA,INGENIERIA ELECTRONICA,1229190,10547081412026A,PROFESIONAL


In [92]:
df_final_final.to_excel(r'C:\Users\alejandro_arboledas\Downloads\BASE_MERCADEO_Y_PUBLICIDAD.xlsx')